# 01 - Versionando prompts com MLflow

Caso prático: um prompt de **resumo de vendas** guardado em YAML. Vamos registrar a primeira versão no MLflow, usá-la, alterar o prompt, registrar a segunda versão e, por fim, usar qualquer uma das duas.

Fluxo:

```
prompts/resumo_vendas.yaml (v1) → registrar → MLflow: versão 1 → usar
        │ pequena alteração
prompts/resumo_vendas.yaml (v2) → registrar → MLflow: versão 2
                                              └── carregar 1 ou 2 pelo número (ou alias) → usar
```

- **YAML como fonte**: o arquivo guarda as mensagens (`system` e `user`) e uma mensagem de commit. É ele que se edita.
- **Prompt registry**: o MLflow guarda cada registro como uma versão imutável. Editar o arquivo não altera versões antigas.
- **Backend local**: o Prompt Registry precisa de um banco. Usamos SQLite (`mlflow.db`), em vez da pasta `mlruns`, que não suporta registro de prompts.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto. O MLflow grava o banco `mlflow.db` na pasta do notebook.

In [1]:
import os
from pathlib import Path

import mlflow
import mlflow.genai
import yaml
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from mlflow import MlflowClient

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")

NOME_PROMPT = "resumo-vendas"
ARQUIVO_YAML = Path("prompts/resumo_vendas.yaml")
ARQUIVO_YAML.parent.mkdir(exist_ok=True)

2026/11/16 21:12:41 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/leonardo/Documentos/github/loop_engineering_study/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 2. Prompt em YAML (versão 1)

O YAML tem três campos: `commit` (descrição da mudança), `system` e `user`. O campo `user` usa `{{ tema }}` e `{{ dados }}` como variáveis, no formato que o MLflow reconhece.

In [2]:
YAML_V1 = """
commit: "Versão inicial do resumo de vendas"
system: |
  Você é um analista de vendas. Escreva em português e use apenas os dados fornecidos.
user: |
  Tema: {{ tema }}
  Dados: {{ dados }}

  Escreva um resumo em um parágrafo.
"""

ARQUIVO_YAML.write_text(YAML_V1, encoding="utf-8")
print(ARQUIVO_YAML.read_text(encoding="utf-8"))


commit: "Versão inicial do resumo de vendas"
system: |
  Você é um analista de vendas. Escreva em português e use apenas os dados fornecidos.
user: |
  Tema: {{ tema }}
  Dados: {{ dados }}

  Escreva um resumo em um parágrafo.



## 3. Registrando a versão 1

`registrar` lê o YAML, monta as mensagens de chat e registra o resultado no MLflow. O MLflow devolve a versão criada.

In [3]:
def registrar(caminho: Path):
    dados = yaml.safe_load(caminho.read_text(encoding="utf-8"))
    mensagens = [
        {"role": "system", "content": dados["system"]},
        {"role": "user", "content": dados["user"]},
    ]
    return mlflow.genai.register_prompt(
        name=NOME_PROMPT,
        template=mensagens,
        commit_message=dados["commit"],
    )


versao_1 = registrar(ARQUIVO_YAML)
print(f"Registrado: {versao_1.name} versão {versao_1.version}")

2026/11/16 21:12:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/16 21:12:43 INFO mlflow.store.db.utils: Updating database tables


Registrado: resumo-vendas versão 1


## 4. Usando a versão 1

`load_prompt` busca a versão pelo URI `prompts:/<nome>/<versão>`. `format` preenche as variáveis e devolve as mensagens prontas para o LLM.

In [4]:
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), api_key=os.getenv("OPENAI_API_KEY"))

DADOS = "Vendas mensais de 2026 (R$ mil): jan 120, fev 98, mar 143, abr 110."
TEMA = "Vendas do primeiro quadrimestre"


def responder(versao: int) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{NOME_PROMPT}/{versao}")
    mensagens = prompt.format(tema=TEMA, dados=DADOS)
    return llm.invoke(mensagens).content


display(Markdown(f"**Resposta com a versão 1:**\n\n{responder(1)}"))

**Resposta com a versão 1:**

No primeiro quadrimestre de 2026 as vendas mensais (em R$ mil) foram: jan 120, fev 98, mar 143 e abr 110, totalizando R$ 471 mil e com média mensal de R$ 117,75 mil; o pico ocorreu em março (R$ 143 mil) e o menor valor em fevereiro (R$ 98 mil), com diferença de R$ 45 mil entre eles. Houve queda de janeiro para fevereiro de R$ 22 mil, seguida de recuperação em março (+R$ 45 mil) e nova queda em abril (-R$ 33 mil).

## 5. Alterando o prompt (versão 2)

A mudança é pequena: pedir que o resumo tenha até 3 tópicos e destaque o mês de maior e o de menor venda. Editamos o YAML, comparamos com a versão anterior e registramos de novo.

In [5]:
YAML_V2 = YAML_V1.replace(
    '"Versão inicial do resumo de vendas"',
    '"Pede resumo em até 3 tópicos e destaca maior e menor venda"',
).replace(
    "Escreva um resumo em um parágrafo.",
    "Escreva um resumo em até 3 tópicos. Destaque o mês de maior e o de menor venda.",
)

ARQUIVO_YAML.write_text(YAML_V2, encoding="utf-8")

import difflib
print("".join(difflib.unified_diff(YAML_V1.splitlines(True), YAML_V2.splitlines(True), "v1", "v2")))

versao_2 = registrar(ARQUIVO_YAML)
print(f"Registrado: {versao_2.name} versão {versao_2.version}")

--- v1
+++ v2
@@ -1,9 +1,9 @@
 
-commit: "Versão inicial do resumo de vendas"
+commit: "Pede resumo em até 3 tópicos e destaca maior e menor venda"
 system: |
   Você é um analista de vendas. Escreva em português e use apenas os dados fornecidos.
 user: |
   Tema: {{ tema }}
   Dados: {{ dados }}
 
-  Escreva um resumo em um parágrafo.
+  Escreva um resumo em até 3 tópicos. Destaque o mês de maior e o de menor venda.

Registrado: resumo-vendas versão 2


## 6. Listando as versões

Cada versão guarda a mensagem de commit. Assim dá para saber o que mudou sem abrir o código.

In [6]:
client = MlflowClient()
for versao in client.search_prompt_versions(NOME_PROMPT):
    print(f"versão {versao.version}: {versao.commit_message}")

versão 2: Pede resumo em até 3 tópicos e destaca maior e menor venda
versão 1: Versão inicial do resumo de vendas


## 7. Usando as duas versões

Como as versões são imutáveis, a versão 1 continua disponível mesmo depois da alteração. O mesmo código roda com a versão que for pedida.

In [7]:
display(Markdown(f"**Resposta com a versão 1:**\n\n{responder(1)}"))
display(Markdown(f"**Resposta com a versão 2:**\n\n{responder(2)}"))

**Resposta com a versão 1:**

No primeiro quadrimestre de 2026 as vendas somaram R$ 471 mil, com média mensal de R$ 117,75 mil; o mês de maior faturamento foi março (R$ 143 mil) e o menor foi fevereiro (R$ 98 mil). Houve uma queda de janeiro para fevereiro de R$ 22 mil (-18,3%), seguida por alta de fevereiro para março de R$ 45 mil (+45,9%) e nova queda de março para abril de R$ 33 mil (-23,1%), resultando em uma redução líquida de janeiro a abril de R$ 10 mil (-8,3%).

**Resposta com a versão 2:**

- Total do 1º quadrimestre: R$ 471 mil; média mensal: R$ 117,75 mil.
- Mês de maior venda: março — R$ 143 mil. Mês de menor venda: fevereiro — R$ 98 mil.
- Distribuição: dois meses acima da média (janeiro R$ 120 mil, março R$ 143 mil) e dois abaixo (fevereiro R$ 98 mil, abril R$ 110 mil); amplitude entre maior e menor = R$ 45 mil.

## 8. Alias (opcional)

Um alias é um apelido para uma versão, por exemplo `producao`. Quem consome o prompt usa o alias, e quem publica decide qual versão ele aponta. Isso permite trocar a versão em produção sem mudar o código que a usa.

In [8]:
mlflow.genai.set_prompt_alias(NOME_PROMPT, alias="producao", version=1)

prompt_producao = mlflow.genai.load_prompt(f"prompts:/{NOME_PROMPT}@producao")
print(f"Alias 'producao' aponta para a versão {prompt_producao.version}")

mlflow.genai.set_prompt_alias(NOME_PROMPT, alias="producao", version=2)
prompt_producao = mlflow.genai.load_prompt(f"prompts:/{NOME_PROMPT}@producao")
print(f"Alias 'producao' agora aponta para a versão {prompt_producao.version}")

Alias 'producao' aponta para a versão 1
Alias 'producao' agora aponta para a versão 2


## 9. Observações

- **Reexecução**: cada execução do notebook registra novas versões. Para começar do zero, apague `mlflow.db` e a pasta `prompts/`.
- **Mesmo texto, mesma versão?** O registro cria uma versão nova a cada chamada de `register_prompt`, mesmo que o texto seja igual. Por isso o notebook registra só quando há mudança.


## 10. Visualizando os prompts na interface

Depois de executar o notebook, o banco `mlflow.db` já tem os prompts. Para vê-los no navegador:

1. Abra um terminal na pasta do notebook (`notebooks/03-MLFlow`). O caminho `sqlite:///mlflow.db` é relativo a ela, então o terminal precisa estar nessa pasta.
2. Suba a interface com o mesmo banco usado pelo notebook:

   ```bash
   uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
   ```

   O `--project ../..` faz o `uv` usar o ambiente do projeto, onde o MLflow está instalado.
3. Abra `http://127.0.0.1:5000` no navegador e procure a seção **Prompts**. Você verá `resumo-vendas` com as versões 1 e 2, a mensagem de commit de cada uma e o alias `producao`. Ao abrir uma versão, aparece o template registrado.
4. Para parar o servidor, volte ao terminal e pressione `Ctrl+C`.

Se a porta 5000 estiver ocupada (no macOS, por exemplo, o AirPlay usa essa porta), troque por outra, como `--port 5055`.

A interface só lê o banco. Para recomeçar do zero, pare o servidor, apague `mlflow.db` e a pasta `prompts/`, e execute o notebook de novo.